In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
print("Veri şekli:", df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
print("MSZoning benzersiz değerler:", df["MSZoning"].unique())
print("\nMSZoning value_counts():")
print(df["MSZoning"].value_counts())

In [ ]:
neighborhood_mean = df.groupby("Neighborhood")["SalePrice"].mean().sort_values(ascending=False).head(10)
print("Mahalleye göre ortalama fiyat (ilk 10):")
print(neighborhood_mean)

In [ ]:
dummies = pd.get_dummies(df[["MSZoning"]], drop_first=True)
print("pd.get_dummies(MSZoning) örnek şekil:", dummies.shape)
display(dummies.head())

In [ ]:
df["BigGarage"] = np.where(df["GarageArea"] > 500, 1, 0)
print("BigGarage sütunu oluşturuldu. 1 sayısı:", int((df["BigGarage"]==1).sum()))
display(df[["GarageArea", "BigGarage"]].head())

In [ ]:
features = ["GrLivArea", "OverallQual", "GarageArea"]
target = "SalePrice"
missing = [c for c in features + [target] if c not in df.columns]
assert len(missing) == 0, f"Eksik sütun(lar): {missing}"
data = df[features + [target]].copy()
before = data.shape[0]
data = data.dropna(subset=features + [target])
after = data.shape[0]
print(f"Dropna ile {before - after} satır atıldı. Kalan: {after}")
data.describe().T

In [ ]:
for col in features:
    plt.figure()
    plt.scatter(data[col], data[target], alpha=0.5)
    plt.title(f"{col} vs {target}")
    plt.xlabel(col)
    plt.ylabel(target)
    plt.tight_layout()
    plt.show()

In [ ]:
corr = data[features + [target]].corr(numeric_only=True)
print(corr)

In [ ]:
X = data[features].values
y = data[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("Katsayılar (coef_):", dict(zip(features, model.coef_)))
print("Sabit (intercept_):", model.intercept_)
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²: {r2:.4f}")

In [ ]:
plt.figure()
plt.scatter(y_test, y_pred, alpha=0.6)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx])
plt.title("Gerçek vs Tahmin")
plt.xlabel("Gerçek SalePrice")
plt.ylabel("Tahmin SalePrice")
plt.tight_layout()
plt.show()